# 第16章 投资组合优化 —— 配套代码

对应正文 `book/part4/16-portfolio-optimization.md`。依赖内置数据，离线可跑。

**本 notebook 演示内容**：
1. 环境初始化与参数估计（μ、Σ）
2. 组合收益与风险基本公式验证
3. 蒙特卡洛撒点：随机组合散点图
4. 全局最小方差（GMV）解析解与数值解对比
5. 有效前沿扫描（scipy 数值求解）
6. 引入无风险资产：最大夏普（切点）组合
7. 资本市场线（CML）可视化
8. 禁止做空约束：有效前沿对比
9. Ledoit-Wolf 收缩协方差对比
10. 等权基准对比
11. 完整汇总图
12. 习题参考解答

> **提示**：先运行 Cell 1（环境初始化），其余格均依赖该格输出。

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# Cell 1: 环境初始化与参数估计
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import inv
from sklearn.covariance import LedoitWolf

from fds import load_sample_prices, load_market, daily_returns, set_chinese_font

set_chinese_font()

# 加载数据
prices = load_sample_prices()
rets = daily_returns(prices)          # 日收益率 DataFrame
market = load_market()

TRADING_DAYS = 252
N = rets.shape[1]                     # 资产数 = 4
tickers = list(rets.columns)

# 年化参数估计
mu = rets.mean().values * TRADING_DAYS          # 年化期望收益（ndarray）
cov_sample = rets.cov().values * TRADING_DAYS   # 年化协方差矩阵（样本）
rf = float(market['rf_annual'].mean())          # 年化无风险利率

print(f"资产数：{N}，交易日数：{len(rets)}")
print(f"年化无风险利率：{rf:.4f}")
print("\n年化期望收益率：")
for t, m in zip(tickers, mu):
    print(f"  {t}: {m:.4f}")
print("\n年化协方差矩阵（样本）：")
print(pd.DataFrame(cov_sample, index=tickers, columns=tickers).round(6).to_string())

## 14.2 组合收益与风险基本公式验证

对于权重向量 $w$，组合的期望收益与年化波动率分别为：

$$
\mu_p = w^\top\mu, \quad \sigma_p = \sqrt{w^\top\Sigma w}
$$

下面用**等权组合**验证，并演示分散化降险效果。

In [ ]:
# Cell 2: 组合收益与风险公式验证

def portfolio_stats(w, mu_vec, cov_mat):
    """计算组合期望收益、年化波动率、夏普比率"""
    w = np.array(w)
    mu_p = float(w @ mu_vec)
    sigma_p = float(np.sqrt(w @ cov_mat @ w))
    sharpe = (mu_p - rf) / sigma_p if sigma_p > 0 else 0.0
    return mu_p, sigma_p, sharpe

# 等权组合
w_eq = np.ones(N) / N
mu_eq, sigma_eq, sr_eq = portfolio_stats(w_eq, mu, cov_sample)

print("等权组合（1/N）统计：")
print(f"  期望收益（年化）: {mu_eq:.4f}")
print(f"  波动率（年化）: {sigma_eq:.4f}")
print(f"  夏普比率: {sr_eq:.4f}")

# 分散化效果验证
weighted_vol = np.sum(w_eq * np.sqrt(np.diag(cov_sample)))
print(f"\n各资产波动率加权平均（无分散化）: {weighted_vol:.4f}")
print(f"等权组合实际波动率: {sigma_eq:.4f}")
print(f"分散化收益（降低波动率）: {(weighted_vol - sigma_eq) / weighted_vol:.1%}")

# 逐资产贡献分析
print("\n各资产风险贡献（%）：")
sigma_vec = cov_sample @ w_eq
rc = w_eq * sigma_vec / (sigma_eq ** 2)  # 各资产对方差的贡献比例
for t, r in zip(tickers, rc):
    print(f"  {t}: {r:.1%}")

## 14.3 蒙特卡洛撒点

随机生成 5000 个合法组合（权重 $\geq 0$，和为 1），在风险-收益平面上描绘散点图，
直觉感受可行集形状与有效前沿的位置。

In [ ]:
# Cell 3: 蒙特卡洛撒点（固定种子 42）
rng = np.random.default_rng(42)
N_SIM = 5000

sim_sigma = np.zeros(N_SIM)
sim_mu    = np.zeros(N_SIM)
sim_sr    = np.zeros(N_SIM)

for i in range(N_SIM):
    w = rng.dirichlet(np.ones(N))   # 均匀分布于单纯形（禁止做空）
    m, s, sr = portfolio_stats(w, mu, cov_sample)
    sim_sigma[i] = s
    sim_mu[i]    = m
    sim_sr[i]    = sr

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(sim_sigma, sim_mu, c=sim_sr, cmap='viridis',
                alpha=0.4, s=8, label='随机组合（5000个）')
plt.colorbar(sc, ax=ax, label='夏普比率')
ax.scatter(*portfolio_stats(w_eq, mu, cov_sample)[:2],
           marker='D', s=150, color='black', zorder=5, label=f'等权组合')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax.set_xlabel('年化波动率 $\\sigma_p$')
ax.set_ylabel('年化期望收益 $\\mu_p$')
ax.set_title('蒙特卡洛随机组合（颜色=夏普比率）')
ax.legend()
plt.tight_layout()
plt.show()
print(f"最高夏普比率（随机组合）: {sim_sr.max():.4f}")

## 14.4 全局最小方差组合（GMV）

### 解析解

$$
w_{\text{GMV}} = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^\top\Sigma^{-1}\mathbf{1}}
$$

### 数值解

使用 `scipy.optimize.minimize`，约束 $w^\top\mathbf{1}=1$，目标 $\min w^\top\Sigma w$。

In [ ]:
# Cell 4: GMV 解析解 vs 数值解

def gmv_analytical(cov_mat):
    """全局最小方差组合解析解"""
    ones = np.ones(len(cov_mat))
    inv_cov = inv(cov_mat)
    w = inv_cov @ ones
    w /= ones @ w
    return w

def gmv_numerical(cov_mat, allow_short=True):
    """全局最小方差组合数值解"""
    n = len(cov_mat)
    w0 = np.ones(n) / n
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    bounds = None if allow_short else [(0.0, 1.0)] * n
    result = minimize(
        fun=lambda w: w @ cov_mat @ w,
        x0=w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 1000}
    )
    return result.x

w_gmv_ana = gmv_analytical(cov_sample)
w_gmv_num = gmv_numerical(cov_sample, allow_short=True)

print("GMV 权重对比（解析 vs 数值）：")
df_gmv = pd.DataFrame({
    '解析解': w_gmv_ana,
    '数值解': w_gmv_num,
    '差异': np.abs(w_gmv_ana - w_gmv_num)
}, index=tickers)
print(df_gmv.round(8).to_string())
print(f"\n最大绝对差异: {np.abs(w_gmv_ana - w_gmv_num).max():.2e}")

mu_gmv, sigma_gmv, sr_gmv = portfolio_stats(w_gmv_ana, mu, cov_sample)
print(f"\nGMV 组合统计：")
print(f"  期望收益: {mu_gmv:.4f}")
print(f"  波动率: {sigma_gmv:.4f}")
print(f"  夏普比率: {sr_gmv:.4f}")

## 14.5 有效前沿（无约束）

扫描一系列目标收益率 $\mu_0$，对每个 $\mu_0$ 求解最小方差组合，记录对应波动率。

In [ ]:
# Cell 5: 有效前沿数值扫描

def efficient_frontier(mu_vec, cov_mat, n_points=200, allow_short=True):
    """扫描有效前沿，返回 (sigmas, mus, weights_list)"""
    n = len(mu_vec)
    # 确定目标收益范围：从 GMV 收益到最大单资产收益
    w_gmv = gmv_numerical(cov_mat, allow_short=allow_short)
    mu_min = float(w_gmv @ mu_vec)
    mu_max = float(mu_vec.max()) * 1.05
    target_mus = np.linspace(mu_min, mu_max, n_points)

    sigmas, mus, weights = [], [], []
    w0 = np.ones(n) / n
    bounds = None if allow_short else [(0.0, 1.0)] * n

    for target in target_mus:
        constraints = [
            {'type': 'eq', 'fun': lambda w: w.sum() - 1},
            {'type': 'eq', 'fun': lambda w, t=target: w @ mu_vec - t}
        ]
        res = minimize(
            fun=lambda w: w @ cov_mat @ w,
            x0=w0,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'ftol': 1e-12, 'maxiter': 1000}
        )
        if res.success:
            sigmas.append(np.sqrt(res.fun))
            mus.append(target)
            weights.append(res.x)
            w0 = res.x  # warm start

    return np.array(sigmas), np.array(mus), weights

# 无约束有效前沿
ef_sigma, ef_mu, ef_weights = efficient_frontier(mu, cov_sample, allow_short=True)
print(f"有效前沿点数: {len(ef_sigma)}")
print(f"前沿收益范围: [{ef_mu.min():.4f}, {ef_mu.max():.4f}]")
print(f"前沿波动范围: [{ef_sigma.min():.4f}, {ef_sigma.max():.4f}]")

## 14.6 最大夏普组合（切点组合）与资本市场线

最大化 $S(w) = \frac{w^\top\mu - r_f}{\sqrt{w^\top\Sigma w}}$，等价于
最小化 $-S(w)$（负夏普比率）。

In [ ]:
# Cell 6: 最大夏普组合

def max_sharpe(mu_vec, cov_mat, rf_rate, allow_short=True):
    """最大夏普比率组合（切点组合）"""
    n = len(mu_vec)
    w0 = np.ones(n) / n
    bounds = None if allow_short else [(0.0, 1.0)] * n
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]

    def neg_sharpe(w):
        mu_p = float(w @ mu_vec)
        sigma_p = float(np.sqrt(np.maximum(w @ cov_mat @ w, 1e-12)))
        return -(mu_p - rf_rate) / sigma_p

    res = minimize(
        fun=neg_sharpe,
        x0=w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 1000}
    )
    return res.x

# 解析解验证
def max_sharpe_analytical(mu_vec, cov_mat, rf_rate):
    """切点组合解析解（无约束）"""
    excess = mu_vec - rf_rate
    inv_cov = inv(cov_mat)
    w = inv_cov @ excess
    w /= np.ones(len(w)) @ w
    return w

w_tan_num = max_sharpe(mu, cov_sample, rf, allow_short=True)
w_tan_ana = max_sharpe_analytical(mu, cov_sample, rf)

mu_tan, sigma_tan, sr_tan = portfolio_stats(w_tan_num, mu, cov_sample)

print("切点组合权重：")
df_tan = pd.DataFrame({
    '数值解': w_tan_num,
    '解析解': w_tan_ana,
    '差异': np.abs(w_tan_num - w_tan_ana)
}, index=tickers)
print(df_tan.round(8).to_string())
print(f"\n切点组合统计：")
print(f"  期望收益: {mu_tan:.4f}")
print(f"  波动率: {sigma_tan:.4f}")
print(f"  夏普比率: {sr_tan:.4f}")

## 14.7 有效前沿 + CML 综合可视化

资本市场线方程：$\mu = r_f + \text{SR}_{\max} \cdot \sigma$

In [ ]:
# Cell 7: 有效前沿 + CML 完整可视化
fig, ax = plt.subplots(figsize=(11, 7))

# 背景：蒙特卡洛散点
ax.scatter(sim_sigma, sim_mu, c='lightgray', alpha=0.3, s=6, label='随机组合')

# 无约束有效前沿
ax.plot(ef_sigma, ef_mu, 'b-', lw=2.5, label='有效前沿（允许做空）', zorder=3)

# CML
cml_sigma = np.linspace(0, ef_sigma.max() * 1.1, 200)
cml_mu = rf + sr_tan * cml_sigma
ax.plot(cml_sigma, cml_mu, 'k--', lw=1.8, label=f'CML（斜率={sr_tan:.2f}）')
ax.axhline(rf, color='gray', lw=1, linestyle=':', alpha=0.7, label=f'无风险利率 {rf:.2%}')

# 标注特殊组合
ax.scatter(sigma_gmv, mu_gmv, marker='*', s=300, color='red', zorder=5, label=f'GMV组合（σ={sigma_gmv:.2%}）')
ax.scatter(sigma_tan, mu_tan, marker='*', s=300, color='green', zorder=5, label=f'最大夏普（SR={sr_tan:.2f}）')
ax.scatter(sigma_eq, mu_eq, marker='D', s=150, color='black', zorder=5, label='等权组合')

# 标注各资产
colors4 = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, (t, m, s) in enumerate(zip(tickers, mu, np.sqrt(np.diag(cov_sample)))):
    ax.scatter(s, m, marker='^', s=120, color=colors4[i], zorder=4)
    ax.annotate(t, xy=(s, m), xytext=(6, 4), textcoords='offset points', fontsize=9, color=colors4[i])

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax.set_xlabel('年化波动率 $\\sigma_p$', fontsize=12)
ax.set_ylabel('年化期望收益 $\\mu_p$', fontsize=12)
ax.set_title('A股四资产有效前沿与资本市场线（CML）', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

## 14.8 禁止做空约束：有效前沿对比

添加约束 $w_i \geq 0$，观察前沿向右偏移情况。

In [ ]:
# Cell 8: 禁止做空前沿对比

# 计算禁止做空下的有效前沿与特殊组合
ef_sigma_ls, ef_mu_ls, _ = efficient_frontier(mu, cov_sample, allow_short=False)

w_gmv_ls = gmv_numerical(cov_sample, allow_short=False)
mu_gmv_ls, sigma_gmv_ls, _ = portfolio_stats(w_gmv_ls, mu, cov_sample)

w_tan_ls = max_sharpe(mu, cov_sample, rf, allow_short=False)
mu_tan_ls, sigma_tan_ls, sr_tan_ls = portfolio_stats(w_tan_ls, mu, cov_sample)

# 绘制对比图
fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(sim_sigma, sim_mu, c='lightgray', alpha=0.3, s=6, label='随机组合（禁止做空）')

ax.plot(ef_sigma, ef_mu, 'b-', lw=2.5, label='无约束前沿', zorder=3)
ax.plot(ef_sigma_ls, ef_mu_ls, 'r-', lw=2.5, label='禁止做空前沿', zorder=3)

ax.scatter(sigma_gmv, mu_gmv, marker='*', s=250, color='blue', zorder=5, label=f'GMV（无约束）σ={sigma_gmv:.2%}')
ax.scatter(sigma_gmv_ls, mu_gmv_ls, marker='*', s=250, color='red', zorder=5, label=f'GMV（禁止做空）σ={sigma_gmv_ls:.2%}')
ax.scatter(sigma_tan, mu_tan, marker='P', s=220, color='blue', zorder=5, label=f'最大夏普（无约束）SR={sr_tan:.2f}')
ax.scatter(sigma_tan_ls, mu_tan_ls, marker='P', s=220, color='red', zorder=5, label=f'最大夏普（禁止做空）SR={sr_tan_ls:.2f}')

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax.set_xlabel('年化波动率')
ax.set_ylabel('年化期望收益')
ax.set_title('有效前沿对比：无约束 vs 禁止做空')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

# 打印权重对比
print("最大夏普组合权重对比：")
df_compare = pd.DataFrame({
    '无约束': w_tan_num,
    '禁止做空': w_tan_ls
}, index=tickers)
print(df_compare.round(4).to_string())
print(f"\n注：无约束中权重为负的资产被'做空'，禁止做空后被压缩至0")

## 14.9 Ledoit-Wolf 收缩协方差对比

Ledoit-Wolf 收缩公式：$\hat\Sigma_{\text{LW}} = (1-\alpha)\hat\Sigma + \alpha F$

其中 $\alpha$ 为最优收缩系数（由 sklearn 自动计算），$F$ 为对角目标矩阵。

In [ ]:
# Cell 9: Ledoit-Wolf 收缩协方差

# 拟合 Ledoit-Wolf 估计器
lw = LedoitWolf().fit(rets.values)
cov_lw = lw.covariance_ * TRADING_DAYS   # 年化
shrinkage = lw.shrinkage_

print(f"Ledoit-Wolf 最优收缩系数 α = {shrinkage:.4f}")
print("\n样本协方差矩阵 vs LW 收缩协方差矩阵（对角元素对比）：")
df_cov_compare = pd.DataFrame({
    '样本方差': np.diag(cov_sample),
    'LW方差': np.diag(cov_lw),
    '样本波动率': np.sqrt(np.diag(cov_sample)),
    'LW波动率': np.sqrt(np.diag(cov_lw))
}, index=tickers)
print(df_cov_compare.round(6).to_string())

# 基于 LW 协方差的 GMV 与最大夏普
w_gmv_lw = gmv_numerical(cov_lw, allow_short=False)
w_tan_lw = max_sharpe(mu, cov_lw, rf, allow_short=False)

mu_gmv_lw, sigma_gmv_lw, _ = portfolio_stats(w_gmv_lw, mu, cov_lw)
mu_tan_lw, sigma_tan_lw, sr_tan_lw = portfolio_stats(w_tan_lw, mu, cov_lw)

print("\n使用 LW 协方差的 GMV 权重 vs 样本协方差的 GMV 权重：")
df_gmv_compare = pd.DataFrame({
    '样本协方差GMV': w_gmv_ls,
    'LW协方差GMV': w_gmv_lw,
}, index=tickers)
print(df_gmv_compare.round(4).to_string())
print(f"\nLW GMV 波动率: {sigma_gmv_lw:.4f}，样本 GMV 波动率: {sigma_gmv_ls:.4f}")

## 14.10 Ledoit-Wolf 有效前沿对比图

In [ ]:
# Cell 10: LW vs 样本协方差前沿对比

# LW 前沿（禁止做空）
ef_sigma_lw, ef_mu_lw, _ = efficient_frontier(mu, cov_lw, allow_short=False)

fig, ax = plt.subplots(figsize=(11, 7))
ax.plot(ef_sigma_ls, ef_mu_ls, 'r-', lw=2.5, label='样本协方差前沿（禁止做空）')
ax.plot(ef_sigma_lw, ef_mu_lw, 'g-', lw=2.5, linestyle='--', label='LW收缩协方差前沿（禁止做空）')

ax.scatter(sigma_gmv_ls, mu_gmv_ls, marker='*', s=250, color='red', zorder=5,
           label=f'GMV（样本）σ={sigma_gmv_ls:.2%}')
ax.scatter(sigma_gmv_lw, mu_gmv_lw, marker='*', s=250, color='green', zorder=5,
           label=f'GMV（LW）σ={sigma_gmv_lw:.2%}')

ax.scatter(sigma_eq, mu_eq, marker='D', s=150, color='black', zorder=5, label='等权组合')

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax.set_xlabel('年化波动率')
ax.set_ylabel('年化期望收益')
ax.set_title('协方差估计方法对有效前沿的影响\n（红色=样本协方差，绿色=LW收缩）')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

## 14.11 风险平价组合

等风险贡献（ERC）要求 $RC_i = RC_j$ 对所有 $i,j$，即每只资产对组合总方差的贡献相等。

优化目标：
$$
\min_w \sum_{i=1}^N\sum_{j=1}^N\left(RC_i - RC_j\right)^2,\ w_i>0,\ w^\top\mathbf{1}=1
$$

In [ ]:
# Cell 11: 风险平价（等风险贡献）组合

def risk_parity(cov_mat):
    """求解等风险贡献（ERC）组合，使用对数障碍法"""
    n = len(cov_mat)

    def risk_budget_objective(w):
        w = np.array(w)
        sigma2 = float(w @ cov_mat @ w)
        marginal = cov_mat @ w
        rc = w * marginal / sigma2           # 各资产风险贡献比例
        target = np.ones(n) / n              # 目标：均等
        return float(np.sum((rc - target) ** 2))

    w0 = np.ones(n) / n
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = [(1e-6, 1.0)] * n

    result = minimize(
        fun=risk_budget_objective,
        x0=w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-14, 'maxiter': 2000}
    )
    return result.x

w_rp = risk_parity(cov_sample)
mu_rp, sigma_rp, sr_rp = portfolio_stats(w_rp, mu, cov_sample)

# 验证风险贡献均等
marginal = cov_sample @ w_rp
sigma_rp2 = w_rp @ cov_sample @ w_rp
rc_rp = w_rp * marginal / sigma_rp2

print("风险平价组合：")
df_rp = pd.DataFrame({
    '权重': w_rp,
    '风险贡献': rc_rp,
    '目标(1/N)': [1/N]*N
}, index=tickers)
print(df_rp.round(4).to_string())
print(f"\n风险平价组合统计：")
print(f"  期望收益: {mu_rp:.4f}")
print(f"  波动率: {sigma_rp:.4f}")
print(f"  夏普比率: {sr_rp:.4f}")

## 14.12 策略汇总对比

In [ ]:
# Cell 12: 策略汇总对比表 + 风险收益散点

strategies = {
    '等权(1/N)': w_eq,
    'GMV(无约束)': w_gmv_ana,
    'GMV(禁做空)': w_gmv_ls,
    'GMV(LW)': w_gmv_lw,
    '最大夏普(无约束)': w_tan_num,
    '最大夏普(禁做空)': w_tan_ls,
    '风险平价': w_rp,
}

rows = []
for name, w in strategies.items():
    m, s, sr = portfolio_stats(w, mu, cov_sample)
    rows.append({'策略': name, '年化收益': f'{m:.2%}', '年化波动': f'{s:.2%}', '夏普比率': f'{sr:.4f}'})

df_summary = pd.DataFrame(rows).set_index('策略')
print("===== 策略对比汇总 =====")
print(df_summary.to_string())

# 权重热图
weight_df = pd.DataFrame({k: v for k, v in strategies.items()}, index=tickers).T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# 左：风险收益散点
cmap_colors = plt.cm.tab10(np.linspace(0, 1, len(strategies)))
for idx, (name, w) in enumerate(strategies.items()):
    m, s, sr = portfolio_stats(w, mu, cov_sample)
    ax1.scatter(s, m, s=200, color=cmap_colors[idx], zorder=5, label=name, edgecolors='white', lw=1.5)

ax1.plot(ef_sigma, ef_mu, 'b-', lw=1.5, alpha=0.5, label='无约束前沿')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax1.set_xlabel('年化波动率')
ax1.set_ylabel('年化期望收益')
ax1.set_title('各策略风险收益分布')
ax1.legend(fontsize=8, loc='lower right')

# 右：权重热图
im = ax2.imshow(weight_df.values, cmap='RdYlGn', aspect='auto', vmin=-0.5, vmax=1.0)
ax2.set_xticks(range(N))
ax2.set_xticklabels(tickers)
ax2.set_yticks(range(len(strategies)))
ax2.set_yticklabels(list(strategies.keys()), fontsize=9)
ax2.set_title('各策略权重热图（绿=多，红=空）')
plt.colorbar(im, ax=ax2, label='权重')

# 标注数值
for i in range(len(strategies)):
    for j in range(N):
        ax2.text(j, i, f'{weight_df.values[i,j]:.2f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 14.13 习题参考解答

In [ ]:
# Cell 13: 习题 14.1 ——手动计算三资产组合统计
print("="*55)
print("习题 14.1：三资产组合期望收益与波动率")
print("="*55)

mu_ex = np.array([0.12, 0.18, 0.08])
cov_ex = np.array([
    [0.04,  0.012, 0.008],
    [0.012, 0.09,  0.015],
    [0.008, 0.015, 0.0225]
])
w_ex = np.array([0.4, 0.4, 0.2])

mu_p_ex  = w_ex @ mu_ex
var_p_ex = w_ex @ cov_ex @ w_ex
std_p_ex = np.sqrt(var_p_ex)
sum_check = w_ex.sum()

print(f"(a) 组合期望收益 μ_p = {mu_p_ex:.4f} = {mu_p_ex:.2%}")
print(f"(a) 组合方差 σ²_p  = {var_p_ex:.6f}")
print(f"(a) 组合波动率 σ_p = {std_p_ex:.4f} = {std_p_ex:.2%}")
print(f"(b) 权重和：{sum_check:.4f}（应等于1，验证通过={abs(sum_check-1)<1e-10}）")

In [ ]:
# Cell 14: 习题 14.2 ——解析解 vs 数值解 GMV，画有效前沿
print("="*55)
print("习题 14.2：GMV 解析 vs 数值 对比，有效前沿")
print("="*55)

w_gmv_a2 = gmv_analytical(cov_sample)
w_gmv_n2 = gmv_numerical(cov_sample, allow_short=True)
max_diff = np.abs(w_gmv_a2 - w_gmv_n2).max()

print("GMV权重（解析 vs 数值）：")
for t, wa, wn in zip(tickers, w_gmv_a2, w_gmv_n2):
    print(f"  {t}: 解析={wa:.8f}, 数值={wn:.8f}, 差={abs(wa-wn):.2e}")
print(f"最大绝对差异：{max_diff:.2e}（< 1e-6 验证通过={max_diff<1e-6}）")

# 100 条扫描点的有效前沿
ef_s2, ef_m2, _ = efficient_frontier(mu, cov_sample, n_points=100, allow_short=True)

fig, ax = plt.subplots(figsize=(9,6))
ax.plot(ef_s2, ef_m2, 'b-', lw=2, label='有效前沿（100点）')
ax.scatter(sigma_gmv, mu_gmv, marker='*', s=300, color='red', zorder=5, label='GMV')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax.set_xlabel('年化波动率')
ax.set_ylabel('年化期望收益')
ax.set_title('习题14.2：100点扫描有效前沿')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15: 习题 14.3 ——禁止做空约束的影响与负权重分析
print("="*55)
print("习题 14.3：禁止做空约束影响分析")
print("="*55)

print("无约束最大夏普组合权重（可看到负权重=做空）：")
for t, w in zip(tickers, w_tan_num):
    flag = " ← 做空" if w < -0.005 else ""
    print(f"  {t}: {w:.4f}{flag}")

print("\n禁止做空最大夏普组合权重（被压缩至0）：")
for t, w in zip(tickers, w_tan_ls):
    flag = " ← 压缩至0" if w < 0.005 else ""
    print(f"  {t}: {w:.4f}{flag}")

# 同收益水平对比方差
common_mu = min(mu_tan, mu_tan_ls) * 0.95  # 取公共收益水平

def min_var_at_target(mu_vec, cov_mat, target, allow_short):
    n = len(mu_vec)
    w0 = np.ones(n)/n
    bounds = None if allow_short else [(0,1)]*n
    constraints = [
        {'type':'eq','fun': lambda w: w.sum()-1},
        {'type':'eq','fun': lambda w, t=target: w@mu_vec-t}
    ]
    res = minimize(lambda w: w@cov_mat@w, w0, method='SLSQP',
                   bounds=bounds, constraints=constraints,
                   options={'ftol':1e-12,'maxiter':1000})
    return np.sqrt(res.fun) if res.success else np.nan

sigma_unconstrained = min_var_at_target(mu, cov_sample, common_mu, allow_short=True)
sigma_constrained   = min_var_at_target(mu, cov_sample, common_mu, allow_short=False)
print(f"\n目标收益={common_mu:.2%}时：")
print(f"  无约束最小波动率：{sigma_unconstrained:.4f}")
print(f"  禁止做空最小波动率：{sigma_constrained:.4f}")
print(f"  波动率惩罚：{sigma_constrained-sigma_unconstrained:.4f}（约束导致前沿右移验证）")

In [ ]:
# Cell 16: 习题 14.4 ——样本协方差 vs LW 收缩对 GMV 的影响
print("="*55)
print("习题 14.4：LW 收缩 vs 样本协方差对 GMV 的影响")
print("="*55)

# (a) 两种协方差下的 GMV 权重
w_gmv_sample = gmv_numerical(cov_sample, allow_short=False)
w_gmv_lw44   = gmv_numerical(cov_lw, allow_short=False)
l2_dist = np.linalg.norm(w_gmv_sample - w_gmv_lw44)

print("(a) GMV 权重（样本 vs LW）：")
for t, ws, wl in zip(tickers, w_gmv_sample, w_gmv_lw44):
    print(f"  {t}: 样本={ws:.4f}, LW={wl:.4f}, 差={ws-wl:.4f}")
print(f"\n(b) L2 距离 ||w_sample - w_LW||_2 = {l2_dist:.6f}")

# (c) 样本量不足时的影响：只取前 60 天
rets_short = rets.iloc[:60]
cov_short = rets_short.cov().values * TRADING_DAYS
lw_short = LedoitWolf().fit(rets_short.values)
cov_lw_short = lw_short.covariance_ * TRADING_DAYS

w_gmv_short_sample = gmv_numerical(cov_short, allow_short=False)
w_gmv_short_lw     = gmv_numerical(cov_lw_short, allow_short=False)
l2_short = np.linalg.norm(w_gmv_short_sample - w_gmv_short_lw)

print(f"\n(c) 仅使用前60天数据：")
print(f"  LW 收缩系数（全样本）: {shrinkage:.4f}")
print(f"  LW 收缩系数（前60天）: {lw_short.shrinkage_:.4f}  ← 小样本时收缩更强")
print(f"  全样本 L2 距离: {l2_dist:.6f}")
print(f"  小样本 L2 距离: {l2_short:.6f}  ← 小样本差异更大")

In [ ]:
# Cell 17: 习题 14.5 ——风险平价与等权的对比分析
print("="*55)
print("习题 14.5：风险平价 vs 等权 vs GMV 波动率对比")
print("="*55)

# (b) 三种组合波动率对比
print("\n三种组合年化波动率对比：")
comparisons = [
    ('等权(1/N)', w_eq),
    ('GMV(禁做空)', w_gmv_ls),
    ('风险平价', w_rp),
]
for name, w in comparisons:
    m, s, sr = portfolio_stats(w, mu, cov_sample)
    print(f"  {name:12s}: 波动率={s:.2%}, 收益={m:.2%}, 夏普={sr:.4f}")
    print(f"    权重: {dict(zip(tickers, [round(x,4) for x in w]))}")

# 验证风险贡献均等性
print("\n风险平价的风险贡献验证（目标=1/N=25%）：")
sigma_rp_val = w_rp @ cov_sample @ w_rp
for t, w_i, rc_i in zip(tickers, w_rp, rc_rp):
    print(f"  {t}: 权重={w_i:.4f}, 风险贡献={rc_i:.4f} ({rc_i:.1%})")
print(f"  偏差（std of RC）= {np.std(rc_rp):.6f}  ← 越小越均等")

# (c) 相关矩阵越接近单位矩阵，等权越接近风险平价
corr = rets.corr().values
print(f"\n(c) 资产间平均相关系数（非对角）= {(corr.sum()-N)/(N*(N-1)):.4f}")
print("    当相关系数=0且波动率相同时，风险平价 = 等权。")
print("    本例各资产波动率不同，所以权重存在差异。")